In [1]:
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

df = pd.read_csv('resumes_processed.csv')
df = df.dropna(subset=['cleaned_resume'])

model = SentenceTransformer('all-MiniLM-L6-v2')
stop_words = set(stopwords.words('english'))

print("Everything loaded!")
print("Resumes:", len(df))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Everything loaded!
Resumes: 2483


In [2]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

In [3]:
def get_match_score(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    if not cleaned_resume or not cleaned_jd:
        return 0.0
    
    resume_embedding = model.encode(cleaned_resume).reshape(1, -1)
    jd_embedding = model.encode(cleaned_jd).reshape(1, -1)
    
    score = cosine_similarity(resume_embedding, jd_embedding)[0][0]
    return round(float(score * 100), 2)

In [6]:
def get_missing_skills(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    resume_words = set(cleaned_resume.split())
    jd_words = set(cleaned_jd.split())
    
    generic = {'looking', 'experience', 'developer', 'working', 
               'strong', 'ability', 'knowledge', 'years', 
               'good', 'must', 'will', 'also', 'well'}
    
    missing = jd_words - resume_words
    missing = [w for w in missing if len(w) >= 4]
    missing = [w for w in missing if w not in generic]
    
    return sorted(missing)

In [7]:
def analyze_resume(resume_text, jd_text):
    score = get_match_score(resume_text, jd_text)
    missing_skills = get_missing_skills(resume_text, jd_text)
    
    # Score interpretation
    if score >= 60:
        level = "Strong Match"
        advice = "Your resume aligns well with this role."
    elif score >= 40:
        level = "Moderate Match"
        advice = "Consider adding missing skills to strengthen your application."
    elif score >= 20:
        level = "Weak Match"
        advice = "Significant gaps found. Tailor your resume more specifically."
    else:
        level = "Poor Match"
        advice = "This role may not align with your current profile."
    
    return {
        "score": score,
        "level": level,
        "advice": advice,
        "missing_skills": missing_skills
    }

In [11]:
sample_jd = """
Looking for a Python Developer with experience in machine learning,
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
Strong communication and problem solving skills required.
"""

it_resume = df[df['category'] == 'INFORMATION-TECHNOLOGY'].iloc[0]['cleaned_resume']
hr_resume = df[df['category'] == 'HR'].iloc[0]['cleaned_resume']

print("=== IT Resume vs Python JD ===")
result = analyze_resume(it_resume, sample_jd)
print(f"Score: {result['score']}%")
print(f"Level: {result['level']}")
print(f"Advice: {result['advice']}")
print(f"Missing Skills: {result['missing_skills']}")

print("\n=== HR Resume vs Python JD ===")
result2 = analyze_resume(hr_resume, sample_jd)
print(f"Score: {result2['score']}%")
print(f"Level: {result2['level']}")
print(f"Advice: {result2['advice']}")
print(f"Missing Skills: {result2['missing_skills']}")

=== IT Resume vs Python JD ===
Score: 29.11%
Level: Weak Match
Advice: Significant gaps found. Tailor your resume more specifically.
Missing Skills: ['communication', 'docker', 'machinelearning', 'numpy', 'pandas', 'problem', 'python', 'restapis', 'scikitlearn', 'solving', 'tensorflow']

=== HR Resume vs Python JD ===
Score: 15.44%
Level: Poor Match
Advice: This role may not align with your current profile.
Missing Skills: ['communication', 'docker', 'machinelearning', 'numpy', 'pandas', 'problem', 'python', 'restapis', 'scikitlearn', 'solving', 'tensorflow']


In [9]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    
    # Preserve important multi-word terms before cleaning
    text = text.replace('scikit-learn', 'scikitlearn')
    text = text.replace('rest apis', 'restapis')
    text = text.replace('machine learning', 'machinelearning')
    text = text.replace('deep learning', 'deeplearning')
    text = text.replace('natural language processing', 'nlp')
    text = text.replace('computer vision', 'computervision')
    text = text.replace('data science', 'datascience')
    text = text.replace('artificial intelligence', 'artificialintelligence')
    
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

In [10]:
def get_missing_skills(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    
    resume_words = set(cleaned_resume.split())
    jd_words = set(cleaned_jd.split())
    
    generic = {'looking', 'experience', 'developer', 'working', 
               'strong', 'ability', 'knowledge', 'years', 
               'good', 'must', 'will', 'also', 'well',
               'role', 'team', 'skills', 'required', 'plus'}
    
    missing = jd_words - resume_words
    missing = [w for w in missing if len(w) >= 4]
    missing = [w for w in missing if w not in generic]
    
    return sorted(missing)

In [12]:
ml_engine_code = '''
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

# Load model once when file is imported
model = SentenceTransformer("all-MiniLM-L6-v2")
stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("scikit-learn", "scikitlearn")
    text = text.replace("rest apis", "restapis")
    text = text.replace("machine learning", "machinelearning")
    text = text.replace("deep learning", "deeplearning")
    text = text.replace("natural language processing", "nlp")
    text = text.replace("computer vision", "computervision")
    text = text.replace("data science", "datascience")
    text = text.replace("artificial intelligence", "artificialintelligence")
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"[^a-z\\s]", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

def get_match_score(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    if not cleaned_resume or not cleaned_jd:
        return 0.0
    resume_embedding = model.encode(cleaned_resume).reshape(1, -1)
    jd_embedding = model.encode(cleaned_jd).reshape(1, -1)
    score = cosine_similarity(resume_embedding, jd_embedding)[0][0]
    return round(float(score * 100), 2)

def get_missing_skills(resume_text, jd_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_jd = clean_text(jd_text)
    resume_words = set(cleaned_resume.split())
    jd_words = set(cleaned_jd.split())
    generic = {"looking", "experience", "developer", "working",
               "strong", "ability", "knowledge", "years",
               "good", "must", "will", "also", "well",
               "role", "team", "skills", "required", "plus"}
    missing = jd_words - resume_words
    missing = [w for w in missing if len(w) >= 4]
    missing = [w for w in missing if w not in generic]
    return sorted(missing)

def analyze_resume(resume_text, jd_text):
    score = get_match_score(resume_text, jd_text)
    missing_skills = get_missing_skills(resume_text, jd_text)
    if score >= 60:
        level = "Strong Match"
        advice = "Your resume aligns well with this role."
    elif score >= 40:
        level = "Moderate Match"
        advice = "Consider adding missing skills to strengthen your application."
    elif score >= 20:
        level = "Weak Match"
        advice = "Significant gaps found. Tailor your resume more specifically."
    else:
        level = "Poor Match"
        advice = "This role may not align with your current profile."
    return {
        "score": score,
        "level": level,
        "advice": advice,
        "missing_skills": missing_skills
    }
'''

with open('ml_engine.py', 'w') as f:
    f.write(ml_engine_code)

print("ml_engine.py saved!")

ml_engine.py saved!


In [14]:
import importlib.util
spec = importlib.util.spec_from_file_location("ml_engine", "ml_engine.py")
ml = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml)

sample_jd = """
Looking for a Python Developer with experience in machine learning,
scikit-learn, pandas, numpy, SQL, REST APIs, docker, git, tensorflow.
"""
sample_resume = "Python developer skilled in machine learning, tensorflow, keras, sql, git"

result = ml.analyze_resume(sample_resume, sample_jd)
print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'score': 72.65, 'level': 'Strong Match', 'advice': 'Your resume aligns well with this role.', 'missing_skills': ['docker', 'numpy', 'pandas', 'restapis', 'scikitlearn']}
